# Import

In [ ]:
!pip install streamlit
!pip install streamlit_folium

In [ ]:
import glob
import json
from typing import List, Tuple

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
import seaborn as sns
import streamlit as st
from streamlit_folium import st_folium
from folium.plugins import TimestampedGeoJson
import geobleu

# Dataload

In [5]:
def load_datasetA(datasetA_path: str) -> pl.DataFrame:
    """HUNOB2024データセットAを読み込み、緯度・経度、詳細時刻、uidごとの方位情報を追加する"""

    # CSV読み込み
    df_a = pl.read_csv(datasetA_path)
    df_a = df_a.filter(pl.col("uid")==0)

    # 緯度経度情報の追加（例: 緯度 = x*0.005 + 34.497, 経度 = y*0.005 + 136.5）
    add_lat_col = (pl.col("x")*0.005 + 34.497).alias("latitude")
    add_lon_col = (pl.col("y")*0.005 + 136.5).alias("longitude")
    df_a = df_a.with_columns(add_lat_col).with_columns(add_lon_col)

    # 詳細時刻の追加
    # d: 0～74, 0 が 2020/01/05 を表す
    # t: 0～47, 0 が 0時、以降30分間隔
    start_date = pl.lit("2020-01-05T00:00:00").str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S")
    df_a = df_a.with_columns(
        (
            start_date
            + pl.col("d") * pl.duration(days=1)
            + pl.col("t") * pl.duration(minutes=30)
        ).cast(pl.Datetime).alias("datetime")
    )

    # datetime 列を "YYYY-MM-DD HH:mm" の文字列に変換
    # df_a = df_a.with_columns(
    #     pl.col("datetime").dt.strftime("%Y-%m-%d %H:%M").alias("datetime_str")
    # )

    # uid ごとに、日時順にソート
    df_a = df_a.sort(["uid", "datetime"])

    # 同一 uid 内で、前の行の緯度・経度を取得（最初の行はnullになる）
    df_a = df_a.with_columns(
        pl.col("latitude").shift(1).over("uid").alias("prev_lat"),
        pl.col("longitude").shift(1).over("uid").alias("prev_lon")
    )
    
    # 前の地点と現在の地点から方位を計算する関数
    def calculate_bearing(row: dict) -> float:
        prev_lat = row["prev_lat"]
        prev_lon = row["prev_lon"]
        lat = row["latitude"]
        lon = row["longitude"]
        # 最初の行は前の位置がないので None を返す
        if prev_lat is None or prev_lon is None:
            return None
        # 度をラジアンに変換
        lat1 = np.radians(prev_lat)
        lon1 = np.radians(prev_lon)
        lat2 = np.radians(lat)
        lon2 = np.radians(lon)
        delta_lon = lon2 - lon1
        x = np.sin(delta_lon) * np.cos(lat2)
        y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(delta_lon))
        initial_bearing = np.arctan2(x, y)
        initial_bearing = np.degrees(initial_bearing)
        # 0～360度に正規化
        bearing = (initial_bearing + 360) % 360
        return bearing

    # 各行ごとに、前の緯度・経度と現在の緯度・経度から方位を計算
    df_a = df_a.with_columns(
        pl.struct(["prev_lat", "prev_lon", "latitude", "longitude"]).apply(calculate_bearing).alias("bearing")
    )

    return df_a


df_a = load_datasetA("/kaggle/s3storage/01_public/humob-challenge-2024/input/cityA_groundtruthdata.csv.gz")

display(df_a.head(1))
print(len(df_a))

/tmp/ipykernel_5594/2416973802.py:64: DeprecationWarning: `apply` is deprecated. It has been renamed to `map_elements`.
  pl.struct(["prev_lat", "prev_lon", "latitude", "longitude"]).apply(calculate_bearing).alias("bearing")


uid,d,t,x,y,latitude,longitude,datetime,prev_lat,prev_lon,bearing
i64,i64,i64,i64,i64,f64,f64,datetime[μs],f64,f64,f64
0,0,1,79,86,34.892,136.93,2020-01-05 00:30:00,null,null,null


1261


In [1]:
df_a.head()

NameError: name 'df_a' is not defined

# 移動軌跡の可視化

In [7]:
def view_user_trajectory_v1(pl_df:pl.DataFrame, uid = 0)->None:

    # 地図オブジェクトを作成
    m = folium.Map(tiles='OpenStreetMap')

    # 一人分のデータだけをdata_tempに格納する
    data_temp = pl_df.filter(pl.col("uid")==0).filter(pl.col("d")==70)

    # data_tempの順番を日時で昇順ソート
    data_temp = data_temp.sort('datetime', descending=True)

    # data_tempの緯度経度だけを
    data_temp_lat_lon = data_temp.select(["latitude", "longitude"])

    # 緯度経度を配列に格納
    locs = data_temp_lat_lon.to_numpy()

    m.fit_bounds([[data_temp_lat_lon["latitude"].min(),data_temp_lat_lon["longitude"].min()], [data_temp_lat_lon["latitude"].max(),data_temp_lat_lon["longitude"].max()]])

    # 地図に線を追加する。緯度経度の配列をそのまま線として使う
    folium.PolyLine(locs).add_to(m)

    display(m)

view_user_trajectory_v1(df_a)

In [10]:
def view_user_trajectory_v2(pl_df:pl.DataFrame, uid = 0)->None:

    # カスタム三角形マーカーをHTMLとCSSで作成
    def create_triangle_marker(points, rotation, m):
        triangle_html = f"""
        <div style="
            width: 0;
            height: 0;
            border-left: 5px solid transparent;
            border-right: 5px solid transparent;
            border-bottom: 10px solid orange;
            transform: rotate({rotation}deg);
            ">
        </div>
        """
        icon = folium.DivIcon(html=triangle_html)
        folium.Marker(location=points, icon=icon).add_to(m)

    pd_df = pl_df.to_pandas()
    pd_df['points'] = pd_df.apply(lambda x: [x['latitude'], x['longitude']], axis=1)
    # m = folium.Map(tiles='https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg', attr='国土地理院', max_native_zoom=22)
    m = folium.Map(tiles='OpenStreetMap')

    folium.PolyLine(locations=pd_df['points'].to_list(), color="orange").add_to(m)
    # 10レコードごとに向きを追加
    for i,row in pd_df.iterrows():
        create_triangle_marker(row['points'], row['bearing'], m)
    m.fit_bounds(pd_df['points'].to_list())
    display(m)

view_user_trajectory_v2(df_a)


In [3]:
def view_user_trajectory_v3(pl_df:pl.DataFrame, uid = 0)->None:

    pd_df = pl_df.to_pandas()
    pd_df['points'] = pd_df.apply(lambda x: [x['latitude'], x['longitude']], axis=1)

    # m = folium.Map(tiles='https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg', attr='国土地理院')
    m = folium.Map(tiles='OpenStreetMap')
    m.fit_bounds(pd_df['points'].to_list())

    data = []
    for _, row in pd_df.iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["longitude"], row["latitude"]]
            },
            "properties": {
                "time": row["datetime"],
                # "popup": f"Speed: {row['speed']} m/s",
                "icon": "circle",
                "iconstyle": {
                    "fillColor": "blue",
                    "fillOpacity": 0.6,
                    "stroke": "true",
                    "radius": 5
                }
            }
        }
        data.append(feature)

    TimestampedGeoJson(
        {
            "type": "FeatureCollection",
            "features": data,
        },
        transition_time=1,
        add_last_point=False,
        period="PT1S",
        auto_play=False,
        loop=False,
    ).add_to(m)
    display(m)

In [7]:
def view_user_trajectory_v3(df_a):
    m = folium.Map(location=[df_a["latitude"].min(), df_a["longitude"].min()], zoom_start=12)
    data = []
    for row in df_a.iter_rows(named=True):
        # もし datetime 列が Timestamp オブジェクトなら文字列に変換する
        # ここでは、"datetime_str" が存在する場合はそれを使い、なければ str() で変換
        time_str = row.get("datetime_str") or str(row["datetime"])
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["longitude"], row["latitude"]],
            },
            "properties": {
                "time": time_str,  # ここを文字列にしておく
                "popup": f"uid: {row['uid']}<br>{time_str}",
                "icon": "circle",
                "iconstyle": {
                    "fillColor": "red",
                    "fillOpacity": 0.6,
                    "stroke": "true",
                    "radius": 5,
                },
            },
        }
        data.append(feature)

    TimestampedGeoJson(
        {
            "type": "FeatureCollection",
            "features": data,
        },
        transition_time=1,
        add_last_point=False,
        period="PT1S",
        auto_play=False,
        loop=False,
    ).add_to(m)
    
    display(m)

In [8]:
view_user_trajectory_v3(df_a)

# 描画
- https://qiita.com/disr-inc/items/34dd5b88d8330379d2ab

In [44]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import folium
from folium.plugins import TimestampedGeoJson

In [45]:
# カスタム三角形マーカーをHTMLとCSSで作成
def create_triangle_marker(points, rotation, m):
    triangle_html = f"""
    <div style="
        width: 0;
        height: 0;
        border-left: 5px solid transparent;
        border-right: 5px solid transparent;
        border-bottom: 10px solid orange;
        transform: rotate({rotation}deg);
        ">
    </div>
    """
    icon = folium.DivIcon(html=triangle_html)
    folium.Marker(location=points, icon=icon).add_to(m)


In [46]:
df_a_pd = df_a.to_pandas()

In [47]:
df_a_pd

,uid,d,t,x,y,latitude,longitude,datetime,datetime_str
0,0,0,1,79,86,34.892,136.930,2020-01-05 00:30:00,2020-01-05 00:30
1,0,0,2,79,86,34.892,136.930,2020-01-05 01:00:00,2020-01-05 01:00
2,0,0,8,77,86,34.882,136.930,2020-01-05 04:00:00,2020-01-05 04:00
3,0,0,9,77,86,34.882,136.930,2020-01-05 04:30:00,2020-01-05 04:30
4,0,0,19,81,89,34.902,136.945,2020-01-05 09:30:00,2020-01-05 09:30
...,...,...,...,...,...,...,...,...,...
111535170,99999,74,38,119,77,35.092,136.885,2020-03-19 19:00:00,2020-03-19 19:00
111535171,99999,74,39,132,94,35.157,136.970,2020-03-19 19:30:00,2020-03-19 19:30
111535172,99999,74,40,124,105,35.117,137.025,2020-03-19 20:00:00,2020-03-19 20:00
111535173,99999,74,41,121,107,35.102,137.035,2020-03-19 20:30:00,2020-03-19 20:30


In [1]:
df_a_pd['points'] = df_a_pd.apply(lambda x: [x['latitude'], x['longitude']], axis=1)
# m = folium.Map(tiles='https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg', attr='国土地理院', max_native_zoom=22)
m = folium.Map(tiles='OpenStreetMap')
folium.PolyLine(locations=df_a_pd['points'].to_list(), color="orange").add_to(m)
# 10レコードごとに向きを追加
for i,row in df_a_pd[::10].iterrows():
    create_triangle_marker(row['points'], row['bearing'], m)
m.fit_bounds(df_a_pd['points'].to_list())
m

NameError: name 'df_a_pd' is not defined

In [ ]:
# m = folium.Map(tiles='https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg', attr='国土地理院')
m = folium.Map(tiles='OpenStreetMap')

m.fit_bounds(df_trip['points'].to_list())

data = []
for _, row in df_trip.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["longitude"], row["latitude"]]
        },
        "properties": {
            "time": row["measurement_ms"],
            "popup": f"Speed: {row['speed']} m/s",
            "icon": "circle",
            "iconstyle": {
                "fillColor": "blue",
                "fillOpacity": 0.6,
                "stroke": "true",
                "radius": 5
            }
        }
    }
    data.append(feature)

TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": data,
    },
    transition_time=1,
    add_last_point=False,
    period="PT1S",
    auto_play=False,
    loop=False,
).add_to(m)
m